# MAGs Dereplication

MAGs often contain redundancies due to multiple similar genomes being recovered from different samples or assemblies. Dereplication is the process of identifying and keeping only representative genomes from highly similar MAGs. This reduces redundancy, simplifies downstream analyses, and avoids overcounting similar genomes in diversity or abundance studies.

**All the following codes were run on Euler. So they cannot be run on Jupyterhub.**


## 1. Archaea


Compute MinHash signatures for the filtered bacterial MAGs MinHash provides a fast sketch of genome content for comparison.


**Parameter:**
- ksize=105: larger k-mers reduce random matches and better capture genome-level similarity.
- scaled=100: controls the number of hashes kept. Lower values retain more detail for more sensitive comparisons.

In [ ]:
mosh sourmash compute \
    --i-sequence-file $data_dir/mags_filtered_archaea_50.qza \
    --p-ksizes 105 \
    --p-scaled 100 \
    --o-min-hash-signature $data_dir/mags-hashes-archaea.qza

Next, we compare the MinHash signatures to generate a distance matrix. The ksize needs to match the kmer size used in the compute command.

In [ ]:
mosh sourmash compare \
    --i-min-hash-signature $data_dir/mags-hashes-archaea.qza \
    --p-ksize 105 \
    --o-compare-output $data_dir/mags-dist-archaea.qza

Finally, we dereplicate the MAGs based on the distance matrix. MAGs with similarity above the threshold (0.7) are considered redundant.

In [ ]:
mosh annotate dereplicate-mags \
    --i-mags $data_dir/mags_filtered_archaea_50.qza \
    --i-distance-matrix $data_dir/mags-dist-archaea.qza \
    --p-threshold 0.7 \
    --o-dereplicated-mags $data_dir/mags-derep-archaea.qza \
    --o-table $data_dir/mags-table-derep-archaea.qza

## 2. Bacteria
The following codes repeat the same steps as for the archaeal MAGs, with the same parameters.

In [ ]:
mosh sourmash compute \
    --i-sequence-file $data_dir/mags_filtered_bacteria_50.qza \
    --p-ksizes 105 \
    --p-scaled 100 \
    --o-min-hash-signature $data_dir/mags-hashes-bacteria.qza

In [ ]:
mosh sourmash compare \
    --i-min-hash-signature $data_dir/mags-hashes-bacteria.qza \
    --p-ksize 105 \
    --o-compare-output $data_dir/mags-dist-bacteria.qza

In [ ]:
mosh annotate dereplicate-mags \
    --i-mags $data_dir/mags_filtered_bacteria_50.qza \
    --i-distance-matrix $data_dir/mags-dist-bacteria.qza \
    --p-threshold 0.7 \
    --o-dereplicated-mags $data_dir/mags-derep-bacteria.qza \
    --o-table $data_dir/mags-table-derep-bacteria.qza

## 3. Fungi
The following codes repeat the same steps as for the archaeal and bacterial MAGs, with the same parameters.

In [ ]:
mosh sourmash compute \
    --i-sequence-file $data_dir/mags_filtered_all_fungi.qza \
    --p-ksizes 105 \
    --p-scaled 100 \
    --o-min-hash-signature $data_dir/mags-hashes-fungi.qza

In [ ]:
mosh sourmash compare \
    --i-min-hash-signature $data_dir/mags-hashes-fungi.qza \
    --p-ksize 105 \
    --o-compare-output $data_dir/mags-dist-fungi.qza

In [ ]:
mosh annotate dereplicate-mags \
    --i-mags $data_dir/mags_filtered_all_fungi.qza \
    --i-distance-matrix $data_dir/mags-dist-fungi.qza \
    --p-threshold 0.7 \
    --o-dereplicated-mags $data_dir/mags-derep-fungi.qza \
    --o-table $data_dir/mags-table-derep-fungi.qza

## 4. Merging all dereplicated MAGs

We finally collate all three dereplicated MAGs files for the proceeding steps. 

In [ ]:
qiime types collate-feature-data-mags \
  --i-mags $data_dir/mags-derep-bacteria.qza \
  --i-mags $data_dir/mags-derep-archaea.qza \
  --i-mags $data_dir/mags-derep-fungi.qza \
  --o-collated-mags $data_dir/mags_derep_all_domains.qza